# **1. Importações**

In [2]:
import pandas as pd
import numpy as np

# **2. Carregamento dos Dados**

In [13]:
# Leitura dos arquivos CSV.
pedidos = pd.read_csv("/content/pedidos.csv")
cardapio = pd.read_csv("/content/cardapio.csv")

# **3. EDA**

In [15]:
# Exploração inicial dos dados.
print(pedidos.head())

   ID_Pedido        Data  Cliente_ID            Item  Quantidade  \
0          1  2023-11-16         222  Torta de Limao        23.0   
1          2  2024-05-25         232          Esfiha         NaN   
2          3  2024-12-20         276  Wrap de Frango         7.0   
3          4  2024-10-17         198    Batata Frita         NaN   
4          5  2023-07-06         385          Esfiha        17.0   

   Preco_Unitario  
0           12.93  
1            5.87  
2           25.98  
3           11.62  
4            6.94  


In [16]:
print(pedidos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID_Pedido       450 non-null    int64  
 1   Data            450 non-null    object 
 2   Cliente_ID      450 non-null    int64  
 3   Item            450 non-null    object 
 4   Quantidade      431 non-null    float64
 5   Preco_Unitario  430 non-null    float64
dtypes: float64(2), int64(2), object(2)
memory usage: 21.2+ KB
None


In [17]:
print(pedidos.describe())

        ID_Pedido  Cliente_ID  Quantidade  Preco_Unitario
count  450.000000  450.000000  431.000000      430.000000
mean   225.500000  245.864444   15.902552       17.856581
std    130.048068   88.360604    9.087931       11.601652
min      1.000000  100.000000    1.000000        4.090000
25%    113.250000  169.000000    8.000000        8.920000
50%    225.500000  241.500000   16.000000       13.095000
75%    337.750000  325.750000   24.000000       27.402500
max    450.000000  399.000000   30.000000       46.070000


# **4. Feature Engineering**

In [18]:
# Criação da coluna de receita.
pedidos["Receita_Item"] = pedidos["Quantidade"] * pedidos["Preco_Unitario"]

In [19]:
print(pedidos[["Item","Receita_Item"]].head())

             Item  Receita_Item
0  Torta de Limao        297.39
1          Esfiha           NaN
2  Wrap de Frango        181.86
3    Batata Frita           NaN
4          Esfiha        117.98


# **5. Tratamento de Dados**

In [20]:
# Ajuste de valores ausentes.
pedidos["Quantidade"].fillna(pedidos["Quantidade"].mean(), inplace=True)
pedidos.dropna(subset=["Preco_Unitario"], inplace=True)

/tmp/ipykernel_15151/898605047.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  pedidos["Quantidade"].fillna(pedidos["Quantidade"].mean(), inplace=True)


In [21]:
print(pedidos.isnull().sum())

ID_Pedido          0
Data               0
Cliente_ID         0
Item               0
Quantidade         0
Preco_Unitario     0
Receita_Item      18
dtype: int64


# **6. Agregações**

In [22]:
# Agrupamento por item.
agregacoes = pedidos.groupby("Item").agg(
    Quantidade_Total=("Quantidade","sum"),
    Receita_Total=("Receita_Item","sum")
).reset_index()

In [23]:
print(agregacoes.head())

            Item  Quantidade_Total  Receita_Total
0     Acai 300ml        402.902552        6249.76
1   Agua Mineral        327.902552        1392.59
2   Batata Frita        323.902552        3785.57
3        Brownie        465.902552        4440.02
4  Cafe Expresso        392.000000        2657.71


In [24]:
# Top 5
print(agregacoes.sort_values("Quantidade_Total",ascending=False).head(5))

             Item  Quantidade_Total  Receita_Total
7      Hamburguer        510.805104       10859.02
3         Brownie        465.902552        4440.02
0      Acai 300ml        402.902552        6249.76
4   Cafe Expresso        392.000000        2657.71
16  Sushi 8 pecas        378.902552       11552.58


In [25]:
print(agregacoes.sort_values("Receita_Total",ascending=False).head(5))

               Item  Quantidade_Total  Receita_Total
9   Pizza Calabresa        372.805104       14545.57
10  Pizza Mussarela        349.000000       14148.69
16    Sushi 8 pecas        378.902552       11552.58
7        Hamburguer        510.805104       10859.02
12    Salada Caesar        352.000000        9927.96


# **7. Análise Temporal**

In [26]:
# Receita por mês.
pedidos["Data"] = pd.to_datetime(pedidos["Data"])
pedidos["Mes"] = pedidos["Data"].dt.month

In [27]:
print(pedidos.groupby("Mes")["Receita_Item"].sum())

Mes
1      9830.11
2     16011.54
3      7983.08
4      8296.86
5     11000.39
6      8084.72
7      9881.83
8     12691.00
9     14309.65
10     5143.68
11     6901.48
12     7847.99
Name: Receita_Item, dtype: float64


# **8. Merge**

In [28]:
# Integração com cardápio.
df = pedidos.merge(cardapio, on="Item", how="left")

In [29]:
print(df.groupby("Categoria")["Receita_Item"].sum())

Categoria
Bebidas       8773.91
Doces        21209.39
Oriental     21067.22
Salgados     50392.77
Saudaveis    16539.04
Name: Receita_Item, dtype: float64


# **9. Filtros**

In [30]:
# Salgados com quantidade > 10.
print(df[(df["Categoria"]=="Salgados") & (df["Quantidade"]>10)].head())

    ID_Pedido       Data  Cliente_ID          Item  Quantidade  \
1           2 2024-05-25         232        Esfiha   15.902552   
3           4 2024-10-17         198  Batata Frita   15.902552   
4           5 2023-07-06         385        Esfiha   17.000000   
5           6 2023-02-04         260      Empanada   16.000000   
14         15 2023-02-03         103  Batata Frita   14.000000   

    Preco_Unitario  Receita_Item  Mes Categoria  Preco_Base  
1             5.87           NaN    5  Salgados         6.5  
3            11.62           NaN   10  Salgados        12.5  
4             6.94        117.98    7  Salgados         6.5  
5             9.48        151.68    2  Salgados         8.9  
14           13.13        183.82    2  Salgados        12.5  


# **10. KPIs**

In [31]:
# Indicadores principais.
print("Receita Total:", df["Receita_Item"].sum())

Receita Total: 117982.33000000002


In [32]:
print("Itens Vendidos:", df["Quantidade"].sum())

Itens Vendidos: 6833.245939675175


In [33]:
print("Ticket Médio:", df.groupby("Data")["Receita_Item"].sum().mean())

Ticket Médio: 337.0923714285715


# **Extra**

In [34]:
# Percentis de preço e quantidade.
print(np.percentile(df["Preco_Unitario"], [25,50,75]))

[ 8.92   13.095  27.4025]


In [35]:
print(np.percentile(df["Quantidade"], [25,50,75]))

[ 8. 16. 24.]
